In [0]:
# Databricks notebook source

from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import *

# ===========================================================
# CONFIGURATION
# ===========================================================

CATALOG = "adbsmartdatatdp"
SCHEMA = "bronze"

VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/files"

FILES = {

    "customers.csv": {
        "table": "customers_raw",
        "schema": StructType([
            StructField("customer_id", IntegerType(), False),
            StructField("first_name", StringType(), True),
            StructField("last_name", StringType(), True),
            StructField("email", StringType(), True),
            StructField("phone", StringType(), True),
            StructField("city", StringType(), True),
            StructField("country", StringType(), True),
            StructField("registration_date", DateType(), True)
        ])
    },

    "products.csv": {
        "table": "products_raw",
        "schema": StructType([
            StructField("product_id", IntegerType(), False),
            StructField("product_name", StringType(), True),
            StructField("category", StringType(), True),
            StructField("brand", StringType(), True),
            StructField("price", DoubleType(), True),
            StructField("stock", IntegerType(), True)
        ])
    },

    "branches.csv": {
        "table": "branches_raw",
        "schema": StructType([
            StructField("branch_id", IntegerType(), False),
            StructField("branch_name", StringType(), True),
            StructField("city", StringType(), True),
            StructField("country", StringType(), True)
        ])
    },

    "employees.csv": {
        "table": "employees_raw",
        "schema": StructType([
            StructField("employee_id", IntegerType(), False),
            StructField("first_name", StringType(), True),
            StructField("last_name", StringType(), True),
            StructField("position", StringType(), True),
            StructField("branch_id", IntegerType(), True)
        ])
    },

    "sales.csv": {
        "table": "sales_raw",
        "schema": StructType([
            StructField("sale_id", IntegerType(), False),
            StructField("customer_id", IntegerType(), True),
            StructField("product_id", IntegerType(), True),
            StructField("sale_date", DateType(), True),
            StructField("quantity", IntegerType(), True),
            StructField("unit_price", DoubleType(), True),
            StructField("discount", DoubleType(), True)
        ])
    }

}

# ===========================================================
# SELECT CATALOG
# ===========================================================

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("="*70)
print("LOAD BRONZE LAYER")
print("="*70)

# ===========================================================
# LOAD FILES
# ===========================================================

summary = []

for file_name, metadata in FILES.items():

    full_path = f"{VOLUME}/{file_name}"

    print(f"\nReading: {file_name}")

    df = (
        spark.read
        .schema(metadata["schema"])
        .option("header", True)
        .csv(full_path)
        .withColumn("ingestion_timestamp", current_timestamp())
    )

    records = df.count()

    print(f"Rows read : {records}")

    table = f"{CATALOG}.{SCHEMA}.{metadata['table']}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )

    rows = spark.table(table).count()

    print(f"Rows loaded : {rows}")

    summary.append((metadata["table"], records, rows))

# ===========================================================
# SUMMARY
# ===========================================================

print("\n")
print("="*70)
print("LOAD SUMMARY")
print("="*70)

summary_df = spark.createDataFrame(
    summary,
    ["table", "rows_read", "rows_loaded"]
)

display(summary_df)

# ===========================================================
# VALIDATION
# ===========================================================

print("\n")
print("="*70)
print("VALIDATION")
print("="*70)

for file_name, metadata in FILES.items():

    table = f"{CATALOG}.{SCHEMA}.{metadata['table']}"

    print(f"\n{table}")

    spark.sql(f"""
        SELECT COUNT(*) AS total_records
        FROM {table}
    """).show()

print("\n")
print("="*70)
print("BRONZE LOAD COMPLETED SUCCESSFULLY")
print("="*70)